# 08 · Report and deployment readiness

**Single responsibility:** assemble the model card, state evidence and limitations, and make device validation gates explicit

Run after the preceding numbered notebook unless the inputs already exist. Every generated artifact is written outside the notebook so this stage is reproducible.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/base.yaml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from vww_esp32.config import load_config, resolve_paths, seed_everything

config, ROOT = load_config(ROOT / 'configs/base.yaml')
paths = resolve_paths(config, ROOT)
seed_everything(config['project']['seed'])
ROOT


In [ ]:
import json
import pandas as pd

manifest = pd.read_csv(paths['processed'] / 'manifest.csv')
dataset_summary = {'counts': manifest.groupby('split').size().to_dict(), 'class_counts': {f'{split}_{label}': int(count) for (split, label), count in manifest.groupby(['split', 'label']).size().items()}}
metrics = json.loads((paths['artifacts'] / 'reports' / 'test_metrics.json').read_text())
export_info = json.loads((paths['artifacts'] / 'reports' / 'export_info.json').read_text())
{'dataset': dataset_summary, 'metrics': metrics, 'export': export_info}

In [ ]:
from vww_esp32.reporting import build_model_card, write_model_card

card = build_model_card(config, dataset_summary, metrics, export_info)
model_card_path = write_model_card(card, ROOT / 'MODEL_CARD.md')
print(card)

## ESP32-CAM release gates

Mark these only with measurements from the exact target board and camera.

- [ ] Firmware operator resolver equals notebook 07's operator list.
- [ ] Tensor arena high-water measured with ≥20% free margin.
- [ ] Model flash size and total firmware partition fit.
- [ ] Median and p95 inference latency measured over ≥1,000 frames.
- [ ] RGB565 channel order and quantization checked with a known color target.
- [ ] Device preprocessing compared pixel-for-pixel against a host reference.
- [ ] At least 200 device-captured frames cover target lighting, distance, pose, and empty scenes.
- [ ] Threshold revalidated on target-domain data without touching the COCO test result.
- [ ] Failure behavior and downstream debounce/cooldown are documented.

In [ ]:
release_checks = {
    'full_int8_io': export_info['input']['dtype'] == export_info['output']['dtype'] == 'int8',
    'model_size_budget': export_info['size_bytes'] <= config['export']['max_model_bytes'],
    'held_out_metrics_present': metrics.get('samples', 0) > 0,
    'header_generated': (ROOT / 'firmware/esp32_cam_vww/include' / config['export']['header_filename']).exists(),
    'device_latency_measured': False,
    'tensor_arena_measured': False,
    'device_dataset_evaluated': False,
}
release_checks


A desktop export is **not yet a production deployment**. The last three checks intentionally remain false until firmware measurements and device-captured evaluation are supplied.